# FeatureGraph ARC Prize submission

This offline notebook locates Kaggle's runtime challenge file and an attached `featuregraph-research` source snapshot, runs the solver registry, validates the complete two-attempt contract, and writes `/kaggle/working/submission.json`.

In [ ]:
from pathlib import Path
import importlib.util
import json

kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    challenge_candidates = sorted(
        kaggle_input.rglob("arc-agi_test_challenges.json")
    )
    source_candidates = sorted(
        kaggle_input.rglob("src/featuregraph/utils/_arc_agi.py")
    )
    output_path = Path("/kaggle/working/submission.json")
else:
    challenge_candidates = [
        Path("notebooks/challenges_fixture.json")
    ]
    source_candidates = [
        Path("src/featuregraph/utils/_arc_agi.py")
    ]
    output_path = Path("/tmp/featuregraph-kaggle-submission.json")

assert len(challenge_candidates) == 1, challenge_candidates
assert len(source_candidates) == 1, source_candidates
challenges_path = challenge_candidates[0]
solver_path = source_candidates[0]
challenges_path, solver_path, output_path

In [ ]:
spec = importlib.util.spec_from_file_location(
    "featuregraph_arc_agi",
    solver_path,
)
assert spec is not None and spec.loader is not None
arc_agi = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arc_agi)

In [ ]:
report = arc_agi.run_harness(challenges_path, output_path)
assert output_path.exists()

submission = json.loads(output_path.read_text(encoding="utf-8"))
challenges = arc_agi.load_challenges(challenges_path)
arc_agi.validate_submission(challenges, submission)

{
    "submission_path": str(report["submission_path"]),
    "number_of_tasks": report["number_of_tasks"],
    "number_supported": report["number_supported"],
    "number_fallback": report["number_fallback"],
}

In [ ]:
attempt_pairs = sum(len(items) for items in submission.values())
assert attempt_pairs == sum(
    len(task["test"]) for task in challenges.values()
)
assert all(
    set(attempt) == {"attempt_1", "attempt_2"}
    for task_attempts in submission.values()
    for attempt in task_attempts
)
attempt_pairs